# Introduction to Weights & Biases

This notebook demonstrates how to use Weights & Biases (wandb) for experiment tracking in machine learning projects.

## What is Weights & Biases?

Weights & Biases (W&B) is a platform for tracking machine learning experiments, logging model runs, visualizing metrics, and organizing datasets and artifacts. For computational social science, it is useful because it provides a reproducible record of all model configurations, parameters, outputs, and evaluation results—especially when running many LLM prompts, classifiers, or embeddings at scale.

Weights & Biases helps you:
- Track experiments and hyperparameters
- Visualize metrics in real-time
- Compare different runs
- Save and version models
- Collaborate with your team

## Installation

First, install the wandb library if you haven't already:

In [ ]:
%pip install wandb

### Setup Ollama

Make sure you have Ollama installed and running locally. You can download it from [ollama.ai](https://ollama.ai).

Pull a model (we'll use llama3.2 as it's lightweight):

In [ ]:
import ollama

# Pull the model (this will download it if not already present)
# This is a one-time download

# use llama3.2 for faster performance 
# qwen3:4b for a stronger model but requires more resources
ollama.pull('llama3.2')

ProgressResponse(status='success', completed=None, total=None, digest=None)

## Setup and Login

You'll need to create a free account at [wandb.ai](https://wandb.ai) and get your API key.

Add a new line to your .env file:

`WANDB_API_KEY=your_key_here`

Note that you do not need quotation marks this time.

Then, check if the key can be found by `dotenv` -- make sure you are in the main COMPSS211 folder!

In [7]:
%pwd

'/Users/tomvannuenen/Library/CloudStorage/Dropbox/GitHub/DEV/COMPSS-211/lessons/week12_emerging-tech'

In [2]:
import wandb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import matplotlib.pyplot as plt
import json
import requests
from tqdm import tqdm
import time
from dotenv import load_dotenv
load_dotenv()

import wandb
wandb.login()

wandb: Currently logged in as: tomvannuenen (dlab-llms-evals) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Create Sample Text Dataset

Let's create a simple sentiment classification dataset with movie reviews.

In [3]:
# Sample movie reviews for sentiment classification
data = [
    # EASY POSITIVE - any method should get these
    {"text": "This movie was absolutely fantastic! The acting was superb and the plot kept me engaged throughout.", "label": "positive"},
    {"text": "I loved every minute of it. A masterpiece of cinema that will be remembered for years.", "label": "positive"},
    {"text": "Outstanding performances and brilliant direction. Highly recommend!", "label": "positive"},
    
    # EASY NEGATIVE - any method should get these
    {"text": "Terrible movie. Waste of time and money. The plot made no sense whatsoever.", "label": "negative"},
    {"text": "I couldn't wait for it to end. Boring, predictable, and poorly acted.", "label": "negative"},
    {"text": "Disappointing on every level. The writing was lazy and the pacing was off.", "label": "negative"},
    
    # EASY NEUTRAL - any method should get these
    {"text": "It was okay. Nothing special but not terrible either. Just average.", "label": "neutral"},
    {"text": "A decent movie. Some good moments but also some slow parts.", "label": "neutral"},
    {"text": "Mixed feelings about this one. The acting was good but the plot was weak.", "label": "neutral"},

    # TRICKY: Sarcasm and irony (few-shot should help here!)
    {"text": "Oh great, another two hours of my life I'll never get back.", "label": "negative"},
    {"text": "Absolutely 'brilliant' work here.", "label": "negative"},
    {"text": "I just love when movies have no plot.", "label": "negative"},
    {"text": "Fantastic! Just what I needed to see today... not.", "label": "negative"},
    {"text": "Well that was certainly... a movie.", "label": "negative"},
    {"text": "Thanks for wasting my evening. Really appreciate it.", "label": "negative"},
    
    # TRICKY: Rhetorical questions and understatement
    {"text": "I paid money for this?", "label": "negative"},
    {"text": "This is what passes for entertainment these days?", "label": "negative"},
    {"text": "The movie wasn't bad, I guess.", "label": "neutral"},
    {"text": "Could have been worse, I suppose.", "label": "neutral"},
    
    # TRICKY: Understatement (British style)
    {"text": "Not bad at all, actually.", "label": "positive"},
    {"text": "I wouldn't mind watching this again.", "label": "positive"},
    
    # TRICKY POSITIVE: Enthusiasm expressed through negation
    {"text": "I can't stop thinking about this movie!", "label": "positive"},
    {"text": "There's nothing I didn't love about this film.", "label": "positive"},

    # TRICKY: Backhanded start with positive conclusion
    {"text": "I didn't expect much, but this was genuinely delightful.", "label": "positive"},
    {"text": "Despite the poor trailer, the actual movie was brilliant.", "label": "positive"},

    # TRICKY: Minimal/ambiguous text
    {"text": "It's a movie alright.", "label": "neutral"},
    {"text": "Meh.", "label": "neutral"},
    {"text": "Exists.", "label": "neutral"},
    
    # TRICKY: Mixed signals - requires nuance
    {"text": "The visuals were stunning but I've never been so bored in my life.", "label": "negative"},
    {"text": "Great acting wasted on a terrible script.", "label": "negative"},
    {"text": "Started strong but completely fell apart in the second half.", "label": "negative"},
    {"text": "Not what I expected, but that's not necessarily bad.", "label": "neutral"},
    {"text": "Interesting attempt at something different, though it didn't quite work.", "label": "neutral"},
    
    # TRICKY: Backhanded compliments
    {"text": "It's good... if you have absolutely nothing better to do.", "label": "negative"},
    {"text": "I've seen worse, surprisingly.", "label": "neutral"},
    {"text": "At least it was short.", "label": "negative"},
    {"text": "The best part was when it ended.", "label": "negative"},
]

# Convert to DataFrame
df = pd.DataFrame(data)

## Helper Functions for Ollama

Let's create functions to interact with Ollama's API for classification.

Let's try out the difference between one-shot, few-shot and instructional prompting as well.

In [4]:
def call_ollama(prompt, model="llama3.2", temperature=0.1):
    try:
        response = ollama.chat(
            model=model,
            messages=[
                {'role': 'user', 'content': prompt}
            ],
            options={
                'temperature': temperature
            }
        )
        return response['message']['content'].strip()
    except Exception as e:
        print(f"Error calling Ollama: {e}")
        return None

In [5]:
def classify_text(text, model="llama3.2", temperature=0.1, prompt_template="zero-shot"):
    """
    Classify text sentiment using Ollama.
    
    Args:
        prompt_template options:
        - "zero-shot": Just the task, no examples
        - "few-shot": Examples of tricky cases (sarcasm, ambiguity)
        - "instructional": More explicit instructions about what to look for
    """
    
    if prompt_template == "zero-shot":
        prompt = f"""Classify the sentiment of the following movie review as either "positive", "negative", or "neutral". 
                    Respond with ONLY ONE WORD: either positive, negative, or neutral.

                    Review: {text}

                    Sentiment:"""
    
    elif prompt_template == "few-shot":
        prompt = f"""Classify the sentiment of movie reviews as either "positive", "negative", or "neutral".

                    Examples of tricky cases:
                    Review: "I paid money for this?"
                    Sentiment: negative

                    Review: "Oh great, another two hours I'll never get back."
                    Sentiment: negative

                    Review: "It was okay. Nothing special."
                    Sentiment: neutral

                    Review: "Meh."
                    Sentiment: neutral

                    Now classify this review. Respond with ONLY ONE WORD: positive, negative, or neutral.

                    Review: {text}

                    Sentiment:"""
    
    elif prompt_template == "instructional":
        # More detailed instructions - another classic prompting technique
        prompt = f"""Classify the sentiment of the following movie review as either "positive", "negative", or "neutral".

                    Instructions:
                    - Look for sentiment indicators (great, terrible, okay, meh)
                    - Watch for sarcasm (e.g., "Oh great..." is usually negative)
                    - Rhetorical questions ("I paid for this?") are typically negative
                    - Brief or ambiguous responses ("Meh", "It's a movie") are neutral

                    Respond with ONLY ONE WORD: positive, negative, or neutral.

                    Review: {text}

                    Sentiment:"""
        
    response = call_ollama(prompt, model=model, temperature=temperature)
    
    if response is None:
        return "neutral"
    
    response_lower = response.lower().strip()
    
    if "positive" in response_lower:
        return "positive"
    elif "negative" in response_lower:
        return "negative"
    else:
        return "neutral"

In [6]:
# Test the function
test_text = "This movie was incredible! I loved every second of it."
classify_text(test_text)


'positive'

## Example 1: Basic Experiment Tracking with Ollama

This example shows how to:
- Initialize a wandb run
- Log hyperparameters (model, temperature, prompt template)
- Classify text using Ollama
- Log classification metrics
- Finish the run

In [7]:
# 1. Initialize a W&B run
run = wandb.init(
    project="wandb-ollama-demo",
    name="my-first-run",
    config={
        "model": "llama3.2",
        "temperature": 0.1,
        "prompt_template": "zero-shot"
    }
)

# 2. Get hyperparameters from config
config = wandb.config

# 3. Run classification
predictions = []
true_labels = [item["label"] for item in data]

for item in data:
    pred = classify_text(
        item["text"],
        model=config.model,
        temperature=config.temperature,
        prompt_template=config.prompt_template
    )
    predictions.append(pred)

# 4. Calculate and log metrics
accuracy = accuracy_score(true_labels, predictions)
f1 = f1_score(true_labels, predictions, average='weighted', zero_division=0)

wandb.log({
    "accuracy": accuracy,
    "f1_score": f1
})

# 5. Finish the run
print(f"Accuracy: {accuracy:.3f}, F1: {f1:.3f}")
wandb.finish()


Accuracy: 0.892, F1: 0.893


accuracy,▁
f1_score,▁
accuracy,0.89189
f1_score,0.89268


## Example 2: Logging Plots and Visualizations

Wandb can log matplotlib figures like confusion matrices to help visualize classification performance.

In [9]:
import wandb
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import pandas as pd

# Initialize run
run = wandb.init(
    project="wandb-ollama-demo",
    name="single-run-with-viz",
    config={
        "model": "llama3.2",
        "temperature": 0.1,
        "prompt_template": "few-shot"
    }
)

config = wandb.config

# Run predictions
predictions = []
true_labels = df['label'].tolist()
results_data = []

print("Classifying examples...")
for idx, row in tqdm(df.iterrows(), total=len(df)):
    pred = classify_text(
        row['text'],
        model=config.model,
        temperature=config.temperature,
        prompt_template=config.prompt_template
    )
    predictions.append(pred)
    
    # Store results for table
    results_data.append({
        "text": row['text'][:80] + "...",
        "true_label": row['label'],
        "predicted": pred,
        "correct": "✓" if pred == row['label'] else "✗"
    })
    
    time.sleep(0.1)

# 1. Log metrics
accuracy = accuracy_score(true_labels, predictions)
f1 = f1_score(true_labels, predictions, average='weighted', zero_division=0)
wandb.log({"accuracy": accuracy, "f1_score": f1})

# 2. Create and log confusion matrix
labels = sorted(df['label'].unique())
cm = confusion_matrix(true_labels, predictions, labels=labels)
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels).plot(ax=ax, cmap='Blues')
plt.title('Confusion Matrix')
wandb.log({"confusion_matrix": wandb.Image(fig)})
plt.close()

# 3. Log predictions table
predictions_table = wandb.Table(dataframe=pd.DataFrame(results_data))
wandb.log({"predictions": predictions_table})

# 4. Print summary
print(f"\nAccuracy: {accuracy:.3f}, F1: {f1:.3f}")
print(f"View results at: {run.get_url()}")

wandb.finish()

Classifying examples...


100%|██████████| 37/37 [00:12<00:00,  2.88it/s]



Accuracy: 0.892, F1: 0.893
View results at: https://wandb.ai/dlab-llms-evals/wandb-ollama-demo/runs/qt067ott


accuracy,▁
f1_score,▁
accuracy,0.89189
f1_score,0.89283


## Example 3: Comparing Prompting Strategies

Let's compare different prompt templates and temperatures to see which works best.

In [11]:
import wandb
from sklearn.metrics import accuracy_score, f1_score

# Define configurations to compare
configurations = [
    {"prompt_template": "zero-shot", "temperature": 0.1, "model": "llama3.2"},
    {"prompt_template": "few-shot", "temperature": 0.1, "model": "llama3.2"},
    {"prompt_template": "instructional", "temperature": 0.1, "model": "llama3.2"},
    {"prompt_template": "zero-shot", "temperature": 0.5, "model": "llama3.2"},
    {"prompt_template": "few-shot", "temperature": 0.5, "model": "llama3.2"},
    {"prompt_template": "instructional", "temperature": 0.5, "model": "llama3.2"},
]

# Run each configuration
for config_params in configurations:
    run = wandb.init(
        project="wandb-ollama-demo",
        name=f"{config_params['prompt_template']}-temp{config_params['temperature']}",
        config=config_params
    )
    
    print(f"\nTesting: {config_params['prompt_template']}")
    
    # Get predictions
    predictions = []
    true_labels = df['label'].tolist()
    
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        pred = classify_text(
            row['text'],
            model=config_params['model'],
            temperature=config_params['temperature'],
            prompt_template=config_params['prompt_template']
        )
        predictions.append(pred)
        time.sleep(0.1)
    
    # Calculate and log metrics
    accuracy = accuracy_score(true_labels, predictions)
    f1 = f1_score(true_labels, predictions, average='weighted', zero_division=0)
    
    wandb.log({
        "accuracy": accuracy,
        "f1_score": f1,
    })
    
    print(f"Accuracy: {accuracy:.3f}, F1: {f1:.3f}")
    
    wandb.finish()


Testing: zero-shot


100%|██████████| 37/37 [00:14<00:00,  2.53it/s]

Accuracy: 0.892, F1: 0.893


accuracy,▁
f1_score,▁
accuracy,0.89189
f1_score,0.89268



Testing: few-shot


...

accuracy,▁
f1_score,▁
accuracy,0.81081
f1_score,0.81415



Testing: instructional


100%|██████████| 37/37 [00:12<00:00,  3.00it/s]

Accuracy: 0.865, F1: 0.862


accuracy,▁
f1_score,▁
accuracy,0.86486
f1_score,0.86246


## Key Takeaways

1. **Initialize runs**: Use `wandb.init()` to start tracking experiments
2. **Log config**: Save hyperparameters like model name, temperature, and prompt templates in the `config` parameter
3. **Log metrics**: Use `wandb.log()` to track classification performance (accuracy, F1, precision, recall)
4. **Log visualizations**: Pass matplotlib figures and tables to wandb using `wandb.Image()` and `wandb.Table()`
5. **Track per-example**: Log individual predictions to understand model behavior on specific cases
6. **Compare prompts**: Run multiple experiments with different prompting strategies to find what works best
7. **Finish runs**: Always call `wandb.finish()` to close the run properly

## Using This with Your Own Data

To adapt this notebook for your own NLP classification tasks:

1. Replace the sample movie reviews with your own text data
2. Modify the `classify_text()` function prompts to match your task (e.g., topic classification, stance detection)
3. Adjust the label categories in the prompt templates
4. Try different models (llama3.2, mistral, phi, etc.)
5. Experiment with temperature and prompt engineering strategies

Check out the [wandb documentation](https://docs.wandb.ai/) for more features.